
# Governing Equation
**강좌**: *전산유체역학*

## Euler 2D 방정식
$$
\mathbf{U}_t + \mathbf{F}_x + \mathbf{G}_y = 0.
$$

여기서

$$
\mathbf{U} = \begin{bmatrix} 
\rho \\ \rho u \\ \rho v \\ \rho e_t
\end{bmatrix},
\mathbf{F} = \begin{bmatrix} 
\rho u \\ \rho u^2 + p \\ \rho u v \\ \rho u h_t
\end{bmatrix}
\mathbf{G} = \begin{bmatrix} 
\rho v \\ \rho uv \\ \rho v^2 + p \\ \rho v h_t
\end{bmatrix}
$$

여기서 $\rho, \mathbf{V}=(u, v), p$ 는 각각 밀도, 속도 벡터, 압력이다. $e_t$는 Total energy로 내부 에너지 $e$ 와 운동에너지 $\frac{1}{2} |\mathbf{V}|^2$ 의 합이다. 완전 이상기체 가정시 내부 에너지는 다음과 같다.

$$
e = \frac{p}{\rho (\gamma -1)}.
$$

$h_t$ 는 Total enthalpy 로 다음과 같다.

$$
h_t = h + \frac{1}{2} |\mathbf{V}|^2 = e_t + \frac{p}{\rho}
$$


### Jacobian Matrix
유속 벡터 $\mathbf{F}_n = (\mathbf{F}, \mathbf{G}) \cdot \mathbf{n}$은 다음과 같다.

$$
\mathbf{F}_n = U_n \mathbf{U} + p \begin{bmatrix}
0 \\ n_x \\ n_y \\ u
\end{bmatrix},
$$

여기서 Contravariant velocity $U_n = \mathbf{V} \cdot \mathbf{n}$ 이고, $\mathbf{n}=(n_x, n_y)$ 이다. 이를 이용해서 Jacobian 행렬을 계산하면 아래와 같다.

$$
\mathbf{A} = \frac{\partial \mathbf{F}_n}{\partial \mathbf{U}}
= U_n \mathbf{I} +
\left (
\mathbf{U} + p \begin{bmatrix}
0 \\ 0 \\ 0 \\ 1
\end{bmatrix}
\right ) \frac{\partial U_n}{\partial \mathbf{U}}
+ \begin{bmatrix}
0 \\ n_x \\ n_y \\ U_n
\end{bmatrix} \frac{\partial p}{\partial \mathbf{U}}
$$

여기서 미분 열 벡터는 다음과 같다.

$$
\frac{\partial U_n}{\partial \mathbf{U}} = \frac{1}{\rho}\begin{bmatrix} -U_n & n_x & n_y & 0 \end{bmatrix},
\frac{\partial p}{\partial \mathbf{U}} = (\gamma-1)\begin{bmatrix} \frac{|\mathbf{V}|^2}{2} & -u & -v & 1 \end{bmatrix},
$$

그 결과 $\mathbf{A}$ 를 rank-1 행렬의 합으로 표현할 수 있다.

$$
\mathbf{A} =  U_n \mathbf{I}
+ \rho \begin{bmatrix}
1 \\ u \\ v \\ h_t
\end{bmatrix} \frac{\partial U_n}{\partial \mathbf{U}}
+ \begin{bmatrix}
0 \\ n_x \\ n_y \\ U_n
\end{bmatrix} \frac{\partial p}{\partial \mathbf{U}}
$$

In [1]:
import sympy as sy


# Gamma and normal vector
gamma = sy.Symbol("gamma")
nx, ny = sy.symbols('n_x, n_y')

# Primitive variables
rho, u, v, a = sy.symbols('rho, u, v, a')

# Dynamic pressure, contravariant velocity, total enthalpy
q = sy.symbols('q')
Un = sy.Symbol('U_n')
ht = a**2/(gamma-1) + q

A = Un * sy.eye(4) + sy.Matrix([1, u, v, ht ]) @ sy.Matrix([-Un, nx, ny, 0]).T \
                  + (gamma-1)*(sy.Matrix([0, nx, ny, Un]) @ sy.Matrix([q, -u, -v, 1]).T)
A

Matrix([
[                                             0,                                             n_x,                                             n_y,                     0],
[                    -U_n*u + n_x*q*(gamma - 1),                 U_n - n_x*u*(gamma - 1) + n_x*u,                      -n_x*v*(gamma - 1) + n_y*u,       n_x*(gamma - 1)],
[                    -U_n*v + n_y*q*(gamma - 1),                       n_x*v - n_y*u*(gamma - 1),                 U_n - n_y*v*(gamma - 1) + n_y*v,       n_y*(gamma - 1)],
[U_n*q*(gamma - 1) - U_n*(a**2/(gamma - 1) + q), -U_n*u*(gamma - 1) + n_x*(a**2/(gamma - 1) + q), -U_n*v*(gamma - 1) + n_y*(a**2/(gamma - 1) + q), U_n*(gamma - 1) + U_n]])

#### Eigenvalue and Eigenvectors of the Jacobian matrix
1차원일 때와 같은 방법으로 고유치를 구할 수 있다. 우선 $\lambda_{1,2} = U_n$ 인 경우, Eigenvector $\mathbf{v}_{1,2}$ 가 $\frac{\partial u}{\partial \mathbf{U}}$ 와 $\frac{\partial p}{\partial \mathbf{U}}$ 에 수직이어야 하며, 그 값은 다음과 같다.

$$
\mathbf{v}_1 = \begin{bmatrix}
1 \\ u \\ v \\ \frac{|\mathbf{V}|^2}{2}
\end{bmatrix}
\quad
\mathbf{v}_2 = \begin{bmatrix}
0 \\ -n_y \\ n_x \\ v n_x - u n_y
\end{bmatrix}
$$

다른 Eigenvector 가 $\mathbf{v} = \mathbf{a}_1 + \beta \mathbf{a}_2$ 형태로 표현될 수 있으며 정리하면 다음 2개의 고유치를 구할 수 있다.

$$
\lambda_3 = U_n + a, \lambda_4 = U_n - a
$$

$$
\mathbf{v}_3 = \begin{bmatrix}
1 \\ u + n_x a \\ v + n_y a \\ h_t + U_n a
\end{bmatrix},
\mathbf{v}_4 = \begin{bmatrix}
1 \\ u - n_x a \\ v - n_y a \\ h_t - U_n a
\end{bmatrix}
$$

## 격자
- 계산 영역 (Computational Domain)을 나누어 놓은 것을 격자 (Grid or mesh) 라고 한다.

- 종류

    - 정렬 격자: 컴퓨터 상에 Array와 일치하도록 나누어놓은 격자
 
        - Multi-block 격자: 공간을 여러개의 정렬 격자를 맞대서 만든 격자
        
        :::{figure-md} str_grd
        <img src="https://upload.wikimedia.org/wikipedia/commons/e/ef/Block_structured_grid.JPG" alt="str-grd">

        Structured grid (from Wikipedia)
        :::           
      
    - 비정렬 격자: 계산영역을 불규칙한 요소로 나누어놓은 격자
    
        :::{figure-md} ustr_grd
        <img src="https://upload.wikimedia.org/wikipedia/commons/0/03/Fitted_mesh.png" alt="ustr-grd">

        Unstructured grid (from Wikipedia)
        :::               
 
    - 중첩 (Overset) 격자: 격자의 일부가 다른 격자와 중첩되어서 만든 격자
 
        - 이동하는 물체
 
    - 기타: Cartesian grid, Sliding mesh 등
 
- 격자 형식: 다양한 격자 형식이 있으며 해석자 마다 고유의 양식도 있음. 공개된 포맷 중 가장 유명한 것은 Plot3D, [CGNS](https://cgns.github.io) 등이 있다.

### Plot3D format
- NASA에서 개발한 **구조격자(Structured Grid)** 기반 CFD 데이터 포맷
    - 좌표와 유동 해를 분리된 파일로 저장

- ASCII와 Binary 형식 저장 가능
    - Endian 등의 Binary 형식을 유의해야 함.
    
#### 파일 구성
| 파일 | 내용 |
|-----|-----|
| `*.xyz` | 격자 좌표 (Grid file) |
| `*.q` | 유동 해 변수 (Solution file) |

#### 격자파일 (`.xyz` or `.x`)
```raw
nb

ni nj nk      (block 1)

ni nj nk      (block 2)

...

x y z         (for all grid points)
...
```

- nb : block 개수 (multi-block 구조)

- (ni, nj, nk) : 각 block의 격자 크기

- 좌표는 i → j → k 순서로 저장

#### Solution 파일 (`.q`)
```raw
nb

ni nj nk      (block 1)

...

gamma Re Mach time

q1 q2 q3 q4 q5
...
```

- `gamma`, `Re`, `Mach`, `time` : 비열비, 레이놀즈 수, 마하수, 시간 (비정상 계산시)

- `q` : 보존 변수 (3차원)

 
#### Plot3D 입출력 함수 구현
- [NASA Plot3D Utilities](https://nasa.github.io/Plot3D_utilities/_build/html) 참고

In [2]:
# 처음 시작시 압축 해제
!unzip solvers.zip

'unzip'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [3]:
%%writefile solvers/plot3d.py
import numpy as np
import struct


def read_plot3d_b(fname, dtype=np.float64):
    """
    Plot 3D Grid Reader (Single block)
    
    Parameters
    ----------
    fname : string
        격자 이름
    dtype : np.float32 | np.float64
        형식
    """
    with open(fname, 'rb') as f:
        nblocks = struct.unpack("I", f.read(4))[0]
        if nblocks != 1:
            raise ValueError('Only single block can be read')
        
        # Read index
        imax = struct.unpack("I", f.read(4))[0]
        jmax = struct.unpack("I", f.read(4))[0]
        kmax = struct.unpack("I", f.read(4))[0]
        
        # Read grid points
        ndims = 3
        x = np.fromfile(f, dtype)
        
        return x.reshape(ndims, kmax, jmax, imax)
        
        
def write_q(fname, q, mach=1.0, alpha=0.0, reyn=-1, time=0.0):
    """
    Plot 3D solution writer (2D)
    
    Parameter
    ---------
    fname : string
        파일 이름
    q : array
        Conservative variable
    """
    nblocks = 1
    
    if len(q.shape) < 4:
        is_2d = True
    else:
        is_2d = False
    
    # Average    
    if is_2d:
        # Corner point correction
        q[:, 0, 0] = (q[:, 0, 1] + q[:, 1, 0] + q[:, 1, 1])/3
        q[:, 0,-1] = (q[:, 0,-2] + q[:, 1,-1] + q[:, 1,-2])/3
        q[:,-1, 0] = (q[:,-1, 1] + q[:,-2, 0] + q[:,-2, 1])/3
        q[:,-1,-1] = (q[:,-1,-2] + q[:,-2,-1] + q[:,-2,-2])/3
        
        qp = (q[:, :-1, :-1] + q[:, :-1, 1:] + q[:, 1:, :-1] + q[:, 1:, 1:])/4
        kmax = 1
        nvars, jmax, imax = qp.shape
    else:
        qp = (
            q[:, :-1, :-1, :-1] + q[:, :-1, :-1, 1:] + q[:, -1:, 1:, :-1] + q[:,-1:, 1:, 1:] +
            q[:, 1:, :-1, :-1] + q[:, 1:, :-1, 1:] + q[:, 1:, 1:, :-1] + q[:, 1:, 1:, 1:]
        )/8
        nvars, kmax, jmax, imax = qp.shape
        
    with open(fname, 'wb') as f:
        # Write index
        f.write(struct.pack("I", nblocks))
        f.write(struct.pack("I", imax))
        f.write(struct.pack("I", jmax))
        f.write(struct.pack("I", kmax))
        
        f.write(struct.pack("d", mach))
        f.write(struct.pack("d", alpha))
        f.write(struct.pack("d", reyn))
        f.write(struct.pack("d", time))
        
        # Write grid points
        qp[0].tofile(f)
        qp[1].tofile(f)
        qp[2].tofile(f)
        
        if is_2d:
            # Dummpy data for z-momentum
            w = np.zeros_like(qp[0]).tofile(f)
        
        qp[3].tofile(f)
        
        if not is_2d:
            qp[4].tofile(f)

Overwriting solvers/plot3d.py


## Transformation of Governing Equation

### 좌표 변환
각 격자 Cell에서 시간과 공간 좌표 $(t,x,y)$ 를 기준 (Reference) 좌표 $(\tau, \xi, \eta)$ 로 변환하고자 한다.
우선 격자 이동은 고려하지 않는다.

$$
t = \tau, x=x(\xi, \eta), y=y(\xi, \eta)
$$

이 경우 좌표 변환에 따른 Jacobian $\mathcal{J}$은 다음과 같다.

$$
\begin{bmatrix}
d\xi \\ d\eta
\end{bmatrix}
= \mathcal{J}
\begin{bmatrix}
dx \\ dy
\end{bmatrix}, 
\mathcal{J}=\frac{\partial(\xi, \eta)}{\partial(x,y)} = 
\begin{bmatrix}
\xi_x & \xi_y \\
\eta_x & \eta_y
\end{bmatrix}
$$

역 변환 $\mathcal{J}^{-1}$ 은 다음과 같다.

$$
\mathcal{J}^{-1}=\frac{\partial(x,y)}{\partial(\xi, \eta)} = 
\begin{bmatrix}
x_{\xi} & x_{\eta} \\
y_{\xi} & y_{\eta} 
\end{bmatrix}
$$

Jacobian $\mathcal{J}$ 는 역변환을 이용해서 구할 수 있다.

$$
\mathcal{J} = \begin{bmatrix}
\xi_x & \xi_y \\
\eta_x & \eta_y
\end{bmatrix}
= {\mathcal{J}^{-1}}^{-1}
=\frac{1}{|\mathcal{J}^{-1}|}
\begin{bmatrix}
y_{\eta} & -x_{\eta} \\
-y_{\xi} & x_{\xi} 
\end{bmatrix}
= 
\begin{bmatrix}
J y_{\eta} & -J x_{\eta} \\
-J y_{\xi} & J x_{\xi} 
\end{bmatrix}
$$

여기서 $J=|\mathcal{J}|$ 로 $|\mathcal{J}^{-1}|=1/|\mathcal{J}|$ 관계를 적용한 결과이다.

### Transformed Equation

Chain Rule을 이용하고 지배방정식은 다음과 같이 변환할 수 있다.

$$
\frac{\partial \mathbf{U}}{\partial \tau} 
+ \xi_x \frac{\partial \mathbf{F}}{\partial \xi} + \eta_x \frac{\partial \mathbf{F}}{\partial \eta}
+ \xi_y \frac{\partial \mathbf{G}}{\partial \xi} + \eta_y \frac{\partial \mathbf{G}}{\partial \eta} = 0
$$

이 식에 $1/J$를 곱한다. 또한 다음 관계를 이용해서 정리한다.

$$
\frac{\xi_x}{J} \frac{\partial \mathbf{F}}{\partial \xi} 
= \frac{\partial}{\partial \xi} \left( \frac{\xi_x}{J} \mathbf{\mathbf{F}} \right)
- \mathbf{\mathbf{F}} \frac{\partial}{\partial \xi} \left( \frac{\xi_x}{J} \right)
$$

그 결과

$$
\frac{\partial}{\partial \tau} \left ( \frac{\mathbf{U}}{J} \right)
+ \frac{\partial}{\partial \xi} \left ( \frac{\xi_x \mathbf{F} + \xi_y \mathbf{G}}{J} \right )
+ \frac{\partial}{\partial \eta} \left ( \frac{\eta_x \mathbf{F} + \eta_y \mathbf{G}}{J} \right ) \\
- \mathbf{F} \left \{ \frac{\partial}{\partial \xi} \left( \frac{\xi_x}{J} \right)
+ \frac{\partial}{\partial \eta} \left( \frac{\eta_x}{J} \right)\right \}
- \mathbf{G} \left \{ \frac{\partial}{\partial \xi} \left( \frac{\xi_y}{J} \right)
+ \frac{\partial}{\partial \eta} \left( \frac{\eta_y}{J} \right)\right \} =0.
$$

여기서

$$
\frac{\xi_x}{J} = y_{\eta}, \frac{\eta_x}{J}=-y_{\xi} \\
\frac{\xi_y}{J} =-x_{\eta}, \frac{\eta_y}{J}=x_{\xi}
$$

이므로, 마지막 두 항은 0이다. 최종적으로 변환된 방정식은 다음과 같다.

$$
\tilde{\mathbf{U}}_t + \tilde{\mathbf{F}}_{\xi} + \tilde{\mathbf{G}}_{\eta} = 0.
$$

여기서

$$
\tilde{\mathbf{U}} = \frac{\mathbf{U}}{J}, 
\tilde{\mathbf{F}} = \frac{\xi_x \mathbf{F} + \xi_y \mathbf{G}}{J},
\tilde{\mathbf{G}} = \frac{\eta_x \mathbf{F} + \eta_y \mathbf{G}}{J}
$$

이다.

### 구현
위 방정식을 해석하기 위해서는 격자 중심점에서 Jacobian $J$ 와 Face 중심점에서 Metric (ex. $\xi_x/J$) 를 구해아 한다.
역변환을 이용해서 이들을 계산한다.

Cell $(i,j)$ 가 격자점 $(i,j), (i+1,j), (i,j+), , (i+1,j+1)$로 구성되어 있다

- 격자 중심점 $(i,j)$ 에서 Jacobian 크기

$$
J_{i,j} = 1 / (x_{\xi} y_{\eta} - x_{\eta} y_{\xi})
$$

여기서 $x_{\xi}$ 등은 Cell 중심점 기준으로 계산한다. 즉

$$
\begin{align}
x_{\xi} &= ((x_{i+1,j}- x_{i,j}) + (x_{i+1,j+1} - x_{i,j+1}))/2 \\
y_{\xi} &= ((y_{i+1,j}- y_{i,j}) + (y_{i+1,j+1} - y_{i,j+1}))/2 \\
x_{\eta} &= ((x_{i,j+1}- x_{i,j}) + (x_{i+1,j+1} - x_{i+1,j}))/2 \\
y_{\eta} &= ((y_{i,j+1}- y_{i,j}) + (y_{i+1,j+1} - y_{i+1,j}))/2 
\end{align}
$$

- Face 에서 Metric 값
   * $\xi$ 에 수직한 방향 (j 변하는 방향) : $|\nabla \xi| / J$

        $$
        \begin{align}
        (\xi_x / J)_{i+1/2,j} &= y_{i+1,j+1} - y_{i+1, j}, & (\xi_y / J)_{i+1/2,j} &= -(x_{i+1,j+1} - x_{i+1, j}) \\
        (\xi_x / J)_{i-1/2,j} &= y_{i,j+1} - y_{i, j}, & (\xi_y / J)_{i-1/2,j} &= -(x_{i,j+1} - x_{i, j})
        \end{align}
        $$

   * $\eta$ 에 수직한 방향 (i 변하는 방향) : $|\nabla \eta| / J$
   
        $$
        \begin{align}
        (\eta_x / J)_{i,j+1/2} &= - (y_{i+1,j+1} + y_{i, j+1}), 
        & (\eta_y / J)_{i,j+1/2} &= x_{i+1,j+1} + x_{i, j+1} \\
        (\eta_x / J)_{i,j-1/2} &= - (y_{i+1,j} + y_{i, j}), 
        & (\eta_y / J)_{i,j-1/2} &= x_{i+1,j} + x_{i, j}
        \end{align}
        $$

#### Quiz
Jacobian $J$ 와 Metric  $|\nabla \xi| / J$ and $|\nabla \eta| / J$ 의 기하학적인 의미가 무엇인지 확인하시오.

In [4]:
%%writefile solvers/grid.py
import numpy as np

def cell_jacobian(x, y):
    """
    Compute Area of cell using inverse Jacobian
    
    Parameters
    ----------
    x : array
        X coordinate (j, i)
    y : array
        Y coordinate (j, i)
        
    Return
    -------
    jac : array
        Jacobian of cell
    """    
    # Parse dimension
    jmax, imax = x.shape
    imax -= 1
    jmax -= 1
    
    # Allocate array
    jac = np.empty((jmax, imax))
    
    # Compute Jacobian
    for j in range(jmax):
        jp = j + 1
        for i in range(imax):
            ip = i +1 
            xch = ((x[jp, ip] + x[j, ip]) - (x[jp, i] + x[j, i]))/2
            ych = ((y[jp, ip] + y[j, ip]) - (y[jp, i] + y[j, i]))/2
            xet = ((x[jp, ip] + x[jp, i]) - (x[j, ip] + x[j, i]))/2
            yet = ((y[jp, ip] + y[jp, i]) - (y[j, ip] + y[j, i]))/2
            
            jac[j, i] = 1/(xch*yet - ych*xet)
            
    return jac


def face_metric(x, y):
    """
    Compute metric of Face (nabla eta /J)
    
    Parameters
    ----------
    x : array
        X coordinate (j, i)
    y : array
        Y coordinate (j, i)
        
    Return
    -------
    si : array
        Face metric along xi direction
    sj : array
        Face metric along eta direction        
    """
    # Parse dimension of coordinate
    jmax, imax = x.shape
    
    # Allocate arrays for \nabla \xi / J, \nabla \eta / J
    si = np.empty((2, jmax-1, imax))
    sj = np.empty((2, jmax, imax-1))
    
    for j in range(jmax-1):
        jp = j + 1
        for i in range(imax):
            si[0, j, i] =  y[jp, i] - y[j, i]
            si[1, j, i] =-(x[jp, i] - x[j, i])
            
    for j in range(jmax):
        for i in range(imax-1):
            ip = i + 1
            sj[0, j, i] =-(y[j, ip] - y[j, i])
            sj[1, j ,i] =  x[j, ip] - x[j, i]
    
    return si, sj


def cell_x(x, y):
    """
    Compute cell center
    
    Parameters
    ----------
    x : array
        X coordinate (j, i)
    y : array
        Y coordinate (j, i)
        
    Return
    -------
    xc : array
        Cell center of X-dir
    yc : array
        Cell center of Y-dir        
    """
    xc = (x[:-1,:-1] + x[1:,:-1] + x[:-1,1:] + x[1:,1:])/4
    yc = (y[:-1,:-1] + y[1:,:-1] + y[:-1,1:] + y[1:,1:])/4
    
    return xc, yc

Overwriting solvers/grid.py


## 보존항, 원시항, 유속 계산 함수
- 유체 (Fluid)를 다루는 과정에서 보존항, 원시항, 유속을 사용하며 이들을 변환하는 함수를 구성한다.

- 시간 적분은 보존항 (Conservative variables) $\mathbf{U}$ 에 대해 진행한다.

- 계산 과정에서 원시항 (Primitive variables) $\mathbf{U}_p$ 가 필요하며 이를 변환하는 함수를 구성한다.

    $$
    \mathbf{U}_p = [\rho, u, v, p]
    $$

- 유속 벡터 (Flux) $\mathbf{F}_n = (\mathbf{F}, \mathbf{G}) \cdot \mathbf{n}$를 계산하는 함수도 구성한다.

    $$
    \mathbf{F}_n = n_x \mathbf{F} + n_y \mathbf{G} = \begin{bmatrix}
    \rho U_n \\ \rho u U_n + n_x p \\ \rho v U_n + n_y p \\ \rho U_n h_t
    \end{bmatrix}
    $$

    - 여기서 contravariant velocity $U_n =u n_x + v n_y$

In [5]:
%%writefile solvers/fluid.py
import numba as nb
import numpy as np


@nb.jit(nopython=True)
def to_primitive(q, qp, gamma=1.4):
    """
    Converting primitive variables from conservative variables
    
    Parameters
    ----------
    q : vector
        Conservative variables
    qp: vector
        Primitive variables
    gamma : float
        specific heat ratio (default: 1.4)
    """
    rho = q[0]
    u = q[1] / rho
    v = q[2] / rho
    rhoe = (q[3] - 0.5*(q[1]**2 + q[2]**2) /q[0])
    p = (gamma-1)*rhoe
    
    qp[0] = rho
    qp[1] = u
    qp[2] = v
    qp[3] = p
    

@nb.jit(nopython=True)
def to_conservative(qp, q, gamma=1.4):
    """
    Converting conservative variables from primitive variables
    
    Parameters
    ----------
    qp: vector
        Primitive variables
    q : vector
        Conservative variables
    gamma : float
        specific heat ratio (default: 1.4)
    """
    rho, u, v, p = qp
    q[0] = rho
    q[1] = rho*u
    q[2] = rho*v
    rhoe = p / (gamma-1)
    q[3] = rhoe + 0.5*rho*(u**2 + v**2)
    

@nb.jit(nopython=True)    
def to_flux(q, f, n, gamma=1.4):
    """
    Flux calculation
    
    Parameters
    ----------
    q : vector
        Conservative variables
    f : vector
        Flux vector
    n : vector
        Unit normal direction vector
    gamma : float
        specific heat ratio (default: 1.4)

    Return
    ------
    p : float
        Pressure
    un : float
        Contravariant velocity
    """
    rho = q[0]
    u = q[1] / q[0]
    v = q[2] / q[0]
    et = q[3] / q[0]
    
    p = (gamma-1)*rho*(et - 0.5*(u**2 + v**2))
    ht = et + p/rho
    
    # Contravariant velocity
    un = u*n[0] + v*n[1]
    
    f[0] = rho*un
    f[1] = rho*un*u + n[0]*p
    f[2] = rho*un*v + n[1]*p
    f[3] = rho*un*ht
    
    return p, un


@nb.jit(nopython=True)    
def to_pressure(q, n, gamma=1.4):
    """
    Compute pressure
    
    Parameters
    ----------
    q : vector
        Conservative variables
    gamma : float
        specific heat ratio (default: 1.4)

    Return
    ------
    p : float
        Pressure
    un : float
        Contravariant velocity
    """
    rho = q[0]
    u = q[1] / q[0]
    v = q[2] / q[0]
    et = q[3] / q[0]
    
    p = (gamma-1)*rho*(et - 0.5*(u**2 + v**2))
    ht = et + p/rho
    
    # Contravariant velocity
    un = u*n[0] + v*n[1]
    
    return p, un

Overwriting solvers/fluid.py
